# QOS-Style Real-Datasets Wrapper with Modwheel Prefilter (Colab)

This notebook demonstrates a non-invasive wrapper pattern for inserting `mod30`, `mod210`, and `mod2310` before real-dataset processing.

Unlike earlier notebooks that filtered rows after feature construction, this wrapper filters the raw text stream **before TF-IDF vectorization**:

```text
raw text stream
    -> optional modwheel prefilter
    -> feature construction / vectorization
    -> downstream classifier sanity check
```

This is closer to the intended pre-oracle placement in a QOS-style pipeline.

Guardrail: this notebook demonstrates a wrapper integration pattern and workload proxies. It does **not** claim QOS improvement, quantum advantage, or accuracy improvement.

In [ ]:
# Clone your fork and move into repo root.
# If Colab says the folder already exists, restart runtime or comment these two lines after first run.
!git clone https://github.com/thinkthoughts/quantum-oracle-sketching-mod30.git
%cd quantum-oracle-sketching-mod30

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modwheel import STANDARD_WHEELS
from pre_oracle_filter import pre_oracle_candidates_by_name

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score

fig_dir = Path("figures")
data_dir = Path("data")
fig_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)

## 1. Load raw 20 Newsgroups text stream

In [ ]:
RANDOM_STATE = 9423

categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.space",
    "talk.politics.misc",
]

dataset = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_STATE,
)

raw_texts = np.array(dataset.data, dtype=object)
raw_y = np.array(dataset.target)
target_names = dataset.target_names
n_original = len(raw_texts)

n_original, len(raw_y), target_names

## 2. Define QOS-style wrapper pipeline

In [ ]:
def class_balance_l1_shift(y_subset, y_reference, n_classes):
    subset_counts = np.bincount(y_subset, minlength=n_classes)
    ref_counts = np.bincount(y_reference, minlength=n_classes)
    subset_frac = subset_counts / subset_counts.sum()
    ref_frac = ref_counts / ref_counts.sum()
    return float(np.sum(np.abs(subset_frac - ref_frac)))


def evaluate_classifier(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    clf = LinearSVC(random_state=RANDOM_STATE, dual="auto")
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    train_time = time.perf_counter() - t0
    pred = clf.predict(X_test)
    return {
        "train_time_seconds": train_time,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
    }


def run_qos_style_pipeline(texts, y, wheel_name=None):
    """Optional modwheel prefilter before feature construction."""
    t_total0 = time.perf_counter()
    n_before = len(texts)

    if wheel_name is None:
        keep_ids = np.arange(n_before)
        subset_name = "baseline"
    else:
        row_ids = np.arange(n_before)
        keep_ids = np.array(pre_oracle_candidates_by_name(row_ids, wheel_name), dtype=int)
        subset_name = wheel_name

    filtered_texts = list(texts[keep_ids])
    filtered_y = y[keep_ids]

    t_vec0 = time.perf_counter()
    vectorizer = TfidfVectorizer(
        max_features=12000,
        min_df=2,
        stop_words="english",
    )
    X = vectorizer.fit_transform(filtered_texts)
    vectorize_time = time.perf_counter() - t_vec0

    eval_result = evaluate_classifier(X, filtered_y)
    total_time = time.perf_counter() - t_total0

    result = {
        "subset": subset_name,
        "wheel": wheel_name if wheel_name is not None else "none",
        "n_original": n_before,
        "n_retained": len(filtered_texts),
        "retained_fraction": len(filtered_texts) / n_before,
        "reduction_fraction": 1 - len(filtered_texts) / n_before,
        "n_features": X.shape[1],
        "nnz": X.nnz,
        "vectorize_time_seconds": vectorize_time,
        "train_time_seconds": eval_result["train_time_seconds"],
        "total_pipeline_time_seconds": total_time,
        "accuracy": eval_result["accuracy"],
        "balanced_accuracy": eval_result["balanced_accuracy"],
        "class_balance_l1_shift": class_balance_l1_shift(filtered_y, y, len(target_names)),
    }
    return result


run_qos_style_pipeline(raw_texts, raw_y, wheel_name="mod30")

## 3. Run baseline and modwheel wrapper pipelines

In [ ]:
results = []
results.append(run_qos_style_pipeline(raw_texts, raw_y, wheel_name=None))

for wheel_name in STANDARD_WHEELS:
    results.append(run_qos_style_pipeline(raw_texts, raw_y, wheel_name=wheel_name))

results_df = pd.DataFrame(results)
results_csv = data_dir / "06_qos_style_wrapper_results.csv"
results_df.to_csv(results_csv, index=False)
results_df

## 4. Figure 06a — retained fraction

In [ ]:
plot_df = results_df.copy()
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.bar(plot_df["subset"], plot_df["retained_fraction"])
ax.set_title("QOS-Style Wrapper: Retained Raw Input Fraction")
ax.set_xlabel("Subset")
ax.set_ylabel("Retained fraction")
ax.set_ylim(0, 1.05)
ax.grid(True, axis="y", alpha=0.35)
for i, value in enumerate(plot_df["retained_fraction"]):
    ax.text(i, value + 0.02, f"{value:.1%}", ha="center")
fig.tight_layout()
fig_a_svg = fig_dir / "figure_06a_wrapper_retained_fraction.svg"
fig_a_png = fig_dir / "figure_06a_wrapper_retained_fraction.png"
fig.savefig(fig_a_svg)
fig.savefig(fig_a_png, dpi=220)
plt.show()
fig_a_svg, fig_a_png

## 5. Figure 06b — vectorization / preprocessing time

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.bar(plot_df["subset"], plot_df["vectorize_time_seconds"])
ax.set_title("QOS-Style Wrapper: Vectorization Time After Prefiltering")
ax.set_xlabel("Subset")
ax.set_ylabel("Vectorization time (seconds)")
ax.grid(True, axis="y", alpha=0.35)
fig.tight_layout()
fig_b_svg = fig_dir / "figure_06b_wrapper_vectorize_time.svg"
fig_b_png = fig_dir / "figure_06b_wrapper_vectorize_time.png"
fig.savefig(fig_b_svg)
fig.savefig(fig_b_png, dpi=220)
plt.show()
fig_b_svg, fig_b_png

## 6. Figure 06c — balanced accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.plot(plot_df["subset"], plot_df["balanced_accuracy"], marker="o", linewidth=2)
ax.set_title("QOS-Style Wrapper: Balanced Accuracy After Prefiltering")
ax.set_xlabel("Subset")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.35)
fig.tight_layout()
fig_c_svg = fig_dir / "figure_06c_wrapper_balanced_accuracy.svg"
fig_c_png = fig_dir / "figure_06c_wrapper_balanced_accuracy.png"
fig.savefig(fig_c_svg)
fig.savefig(fig_c_png, dpi=220)
plt.show()
fig_c_svg, fig_c_png

## 7. Figure 06d — class-balance shift

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.bar(plot_df["subset"], plot_df["class_balance_l1_shift"])
ax.set_title("QOS-Style Wrapper: Class-Balance L1 Shift")
ax.set_xlabel("Subset")
ax.set_ylabel("L1 shift vs baseline class distribution")
ax.grid(True, axis="y", alpha=0.35)
fig.tight_layout()
fig_d_svg = fig_dir / "figure_06d_wrapper_class_balance_shift.svg"
fig_d_png = fig_dir / "figure_06d_wrapper_class_balance_shift.png"
fig.savefig(fig_d_svg)
fig.savefig(fig_d_png, dpi=220)
plt.show()
fig_d_svg, fig_d_png

## 8. Adapter code block for `real_datasets/20news_svm.py`

In [ ]:
adapter_code = r'''
# Optional modwheel prefilter before feature construction.
# Insert after raw texts / labels are loaded and before vectorization or QOS prep.

from pre_oracle_filter import pre_oracle_candidates_by_name
import numpy as np

def apply_modwheel_text_prefilter(texts, y, wheel_name="mod30"):
    row_ids = np.arange(len(texts))
    keep_ids = np.array(pre_oracle_candidates_by_name(row_ids, wheel_name), dtype=int)
    texts_filtered = [texts[i] for i in keep_ids]
    y_filtered = y[keep_ids]
    return texts_filtered, y_filtered, keep_ids

# Example:
# texts, y, keep_ids = apply_modwheel_text_prefilter(texts, y, "mod30")
'''

adapter_path = data_dir / "06_real_datasets_20news_adapter_snippet.py"
adapter_path.write_text(adapter_code)
print(adapter_code)
adapter_path

## 9. Interpretation

In [ ]:
print("""
Notebook 06 interpretation:

- Filtering is applied before feature construction, closer to the intended pre-oracle position.
- The wrapper reduces the raw input stream before vectorization / downstream processing.
- Vectorization and training-time proxies can be compared against baseline.
- Class-balance shift helps check whether row-ID filtering strongly distorts labels.

Guardrail:
This demonstrates a wrapper integration pattern. It does not claim QOS improvement, quantum advantage, or accuracy improvement.
""")

## 10. Download outputs

In [ ]:
from google.colab import files

for path in [
    "data/06_qos_style_wrapper_results.csv",
    "data/06_real_datasets_20news_adapter_snippet.py",
    "figures/figure_06a_wrapper_retained_fraction.svg",
    "figures/figure_06b_wrapper_vectorize_time.svg",
    "figures/figure_06c_wrapper_balanced_accuracy.svg",
    "figures/figure_06d_wrapper_class_balance_shift.svg",
]:
    files.download(path)